In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F

CLEANED_DATA_DIR = Path("../data/cleaned")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
co2_df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")
eurostat_df = spark.read.csv('../data/cleaned/2c_eurostat_new_passenger_cars_by_type_of_motor_energy.csv', header=True)

eurostat_df.show(10), co2_df.show(10)

In [ ]:
co2_totals = (
    co2_df
    .groupBy("geo", "TIME_PERIOD")
    .agg(F.sum("registrations").cast("double").alias("co2_total"))
    .withColumn("TIME_PERIOD", F.col("TIME_PERIOD").cast("string"))
)

eurostat_totals = (
    eurostat_df
    .filter(F.col("Motor energy").isin("Total", "TOTAL", "total"))
    .select(
        F.col("geo"),
        F.col("TIME_PERIOD").cast("string"),
        F.col("OBS_VALUE").cast("double").alias("eurostat_total")
    )
)

validation_totals = (
    co2_totals.join(eurostat_totals, on=["geo", "TIME_PERIOD"], how="inner")
    .withColumn("difference", F.col("co2_total") - F.col("eurostat_total"))
    .withColumn(
        "percentage_difference",
        F.round((F.abs(F.col("abs_diff")) / F.col("eurostat_total")) * 100, 2)
    )
    .orderBy("geo", "TIME_PERIOD")
)

validation_totals.show(50, truncate=False)